# Etapa 5k — Wiener con el ruido estimado en cada imagen

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** En la etapa 8 el algoritmo funcionó hasta ruido de desvío 5 niveles. Con 10 Matthews cae 0,07 y con 20 cae a 0,41. El Wiener local aplicado a la foto lo recuperaba (+0,05 con σ = 10, +0,24 con σ = 20), pero no se agregó al algoritmo porque en la base no cambiaba nada; quedó como recomendación aplicarlo a fotos con ruido marcado (desvío 14.9). Para que el algoritmo lo haga solo, hace falta saber cuánto ruido tiene cada foto. El Wiener de scipy estima el ruido con el promedio de las varianzas locales: en una foto limpia ese promedio sale alto por los propios vasos, así que la suaviza igual.

**Qué se propone** (`preprocess.wiener_ruido_estimado`, paso "Wiener con ruido estimado"). Sobre el verde a la mitad de resolución, antes del relleno de exudados y de la normalización:
1. **Estimar el desvío del ruido** (`preprocess.estimar_ruido`). Se filtra con la máscara de Immerkær (1996), dos laplacianos restados que anulan el fondo suave y dejan pasar casi solo el ruido, y se toma la mediana del valor absoluto: la desviación mediana absoluta de Donoho y Johnstone (1994), para que los bordes de los vasos no inflen la estimación. En desarrollo estima 0,5 a 0,7 niveles en las fotos limpias, y 2,47, 4,94 y 9,88 con ruido agregado de 5, 10 y 20 (la reducción 2×2 baja el desvío a la mitad: lo esperado es 2,5, 5 y 10).
2. **Wiener local** (Lee, 1980) en ventanas de 5 × 5 (la de la etapa 3), con ese ruido: media + (1 − ruido / varianza local) × (valor − media). En una foto limpia casi no cambia nada (0,1 a 0,2 niveles en promedio, contra 0,9 a 1,1 con el ruido de scipy); en una ruidosa, suaviza el fondo y respeta los vasos.

**Candidatas** (sobre el algoritmo actual: división + relleno de exudados, 28 mapas, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento). Cada una ajusta sus pesos y su valor fijo con las fotos de entrenamiento sin ruido agregado (la base), como en la etapa 8:
- **A.** Sin Wiener (el algoritmo actual).
- **B.** Con Wiener y el ruido estimado.
- **C.** Con el Wiener de la etapa 3 (`restoration.wiener_denoise`, ruido de scipy). Es descriptiva: muestra qué pasa si se filtra siempre. No se puede adoptar.

**Condiciones.** Las fotos de las personas de afuera tal como están, y con ruido gaussiano agregado de desvío 5, 10 y 20 niveles (`degrade.gaussian_noise`, el mismo ruido para las tres candidatas).

**Medidas**, a resolución original, en el campo de visión de la base: Matthews, Dice de línea central, vasos finos (sensibilidad de línea central) y precisión. Descriptivo: el ruido estimado en cada condición.

**Regla de adopción (fijada antes de correr).** B reemplaza a A si
1. en las fotos sin ruido agregado no es inferior a A en Matthews, vasos finos ni Dice de línea central (límite inferior > −0,03), **y**
2. con ruido de desvío 10 (el primer nivel en que el algoritmo dejó de funcionar en la etapa 8) mejora Matthews con un intervalo al 95 % que no incluye el cero.

Si no cumple, queda A.

**Control:** A sin ruido agregado tiene que reproducir el Matthews de A en `05j` (0,7533).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.preprocess import PASO_WIENER_ESTIMADO, PASO_WIENER_LOCAL
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
CANDS = {"A. sin Wiener": C.SEGMENT_BACKGROUND_MODE,
         "B. Wiener con ruido estimado": C.SEGMENT_BACKGROUND_MODE + " + " + PASO_WIENER_ESTIMADO,
         "C. Wiener de la etapa 3": C.SEGMENT_BACKGROUND_MODE + " + " + PASO_WIENER_LOCAL}
ORDER = list(CANDS)
NOISE = (0, 5, 10, 20)
WORKERS = 4
print(CANDS, "|", len(C.SEGMENT_FEATURES), "mapas")

{'A. sin Wiener': 'división + relleno de exudados', 'B. Wiener con ruido estimado': 'división + relleno de exudados + Wiener con ruido estimado', 'C. Wiener de la etapa 3': 'división + relleno de exudados + Wiener local'} | 28 mapas


## Pasada 1: píxeles de entrenamiento (fotos sin ruido agregado)

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, {c: training_pixels(image_id, background=m) for c, m in CANDS.items()}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
samples = {c: {i: P1[i][c] for i in ids} for c in CANDS}
del P1
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 706 s


## Ajuste por vuelta

In [3]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        fit[(k, c)] = (w, choose_value(samples[c], w, tr))
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

,A. sin Wiener,B. Wiener con ruido estimado,C. Wiener de la etapa 3
vuelta 0,0.8614,0.8801,0.8580
vuelta 1,0.8735,0.8761,0.8694
vuelta 2,0.8764,0.8785,0.8562
vuelta 3,0.8619,0.8644,0.8584
vuelta 4,0.8708,0.8730,0.8828


## Pasada 2: las personas de afuera, sin ruido agregado y con ruido 5, 10 y 20

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.degrade import gaussian_noise
    from src.enhance import prepare_maps, upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.preprocess import estimar_ruido
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    clean = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for s in NOISE:
        rgb = clean if s == 0 else gaussian_noise(clean, s, C.SEED + 1000 * s + ids.index(image_id))
        maps, region, _ = prepare_maps(rgb, ["Verde"], 2)
        noise_est = 255 * estimar_ruido(maps["Verde"], region)
        del maps
        auto = detect_fov_robust(rgb)
        for c, mode in CANDS.items():
            w, value = fit[(k, c)]
            (p,), _, _ = feature_scores(rgb, [w], 2, background=mode)
            m = (upscale(p, fov.shape) > value) & auto
            marked = m & fov
            rows.append({"image": image_id, "ruido": s, "candidata": c, "ruido_estimado": noise_est,
                         "matthews": matthews(m, truth, fov), "dice_linea_central": centerline_dice(m, truth, fov),
                         "finos": m[thin].mean(), "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05k_wiener.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 2693 s


## Resultados

In [5]:
print("Ruido estimado (niveles de gris, verde a la mitad de resolución; lo esperado es el agregado / 2):")
display(R[R["candidata"] == ORDER[0]].groupby("ruido")["ruido_estimado"].describe()[["mean", "min", "max"]].round(2))
print("Matthews medio por condición:")
R.pivot_table(index="candidata", columns="ruido", values="matthews").loc[ORDER].round(4)

Ruido estimado (niveles de gris, verde a la mitad de resolución; lo esperado es el agregado / 2):


,mean,min,max
ruido,,,
0,0.54,0.49,0.74
5,2.50,2.47,2.72
10,4.95,4.94,5.19
20,9.90,9.64,10.13


Matthews medio por condición:


ruido,0,5,10,20
candidata,,,,
A. sin Wiener,0.7533,0.7327,0.6482,0.2694
B. Wiener con ruido estimado,0.7525,0.7390,0.7123,0.6207
C. Wiener de la etapa 3,0.7449,0.7282,0.6860,0.5631


In [6]:
def compare(new, s):
    out = []
    sub = R[R["ruido"] == s]
    for meas in ("matthews", "dice_linea_central", "finos", "precision"):
        w = sub.pivot(index="image", columns="candidata", values=meas)
        d = (w[new] - w[ORDER[0]]).to_frame()
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        out.append({"candidata": new[:2], "ruido": s, "medida": meas, "A": w[ORDER[0]].mean(), "nueva": w[new].mean(),
                    "diferencia": d.iloc[:, 0].mean(), "inferior": lo[0], "superior": hi[0],
                    "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
    return out

table = pd.DataFrame([r for c in ORDER[1:] for s in NOISE for r in compare(c, s)]).set_index(["candidata", "ruido", "medida"])
table.round(4)

A   nueva  diferencia  inferior  superior  imágenes que mejoran
candidata ruido medida                                                                                  
B.        0     matthews            0.7533  0.7525     -0.0008   -0.0015   -0.0002                     6
                dice_linea_central  0.7583  0.7568     -0.0016   -0.0023   -0.0009                     1
                finos               0.3534  0.3461     -0.0073   -0.0125   -0.0028                     5
                precision           0.8002  0.8024      0.0022    0.0001    0.0046                    18
          5     matthews            0.7327  0.7390      0.0063    0.0034    0.0090                    27
                dice_linea_central  0.7133  0.7298      0.0165    0.0104    0.0235                    27
                finos               0.2935  0.2751     -0.0184   -0.0288   -0.0078                     6
                precision           0.7989  0.7895     -0.0093   -0.0130   -0.0055                     5
          10    matthews            0.6482  0.7123      0.0641    0.0468    0.0805                    30
                dice_linea_central  0.5938  0.6841      0.0903    0.0713    0.1064                    30
                finos               0.2031  0.2218      0.0187    0.0040    0.0348                    22
                precision           0.7463  0.7776      0.0312    0.0163    0.0441                    23
          20    matthews            0.2694  0.6207      0.3513    0.2962    0.3975                    30
                dice_linea_central  0.2890  0.5719      0.2829    0.2462    0.3152                    30
                finos               0.1024  0.1595      0.0571    0.0407    0.0753                    29
                precision           0.4067  0.7066      0.2999    0.2351    0.3539                    30
C.        0     matthews            0.7533  0.7449     -0.0084   -0.0104   -0.0066                     1
                dice_linea_central  0.7583  0.7486     -0.0097   -0.0147   -0.0042                     6
                finos               0.3534  0.3106     -0.0428   -0.0557   -0.0288                     3
                precision           0.8002  0.7854     -0.0148   -0.0199   -0.0105                     2
          5     matthews            0.7327  0.7282     -0.0046   -0.0105    0.0005                    10
                dice_linea_central  0.7133  0.7277      0.0144    0.0070    0.0229                    25
                finos               0.2935  0.3001      0.0065   -0.0097    0.0243                    17
                precision           0.7989  0.7766     -0.0223   -0.0287   -0.0168                     1
          10    matthews            0.6482  0.6860      0.0378    0.0214    0.0540                    25
                dice_linea_central  0.5938  0.6702      0.0764    0.0609    0.0907                    30
                finos               0.2031  0.2510      0.0479    0.0298    0.0679                    24
                precision           0.7463  0.7634      0.0171    0.0003    0.0308                    19
          20    matthews            0.2694  0.5631      0.2936    0.2438    0.3358                    30
                dice_linea_central  0.2890  0.5224      0.2334    0.2023    0.2604                    30
                finos               0.1024  0.1607      0.0583    0.0407    0.0784                    29
                precision           0.4067  0.6765      0.2699    0.2065    0.3231                    29

In [7]:
b = table.loc["B."]
cond1 = bool(all(b.loc[(0, m), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
                 for m in ("matthews", "finos", "dice_linea_central")))
cond2 = bool(b.loc[(10, "matthews"), "inferior"] > 0)
print("B no inferior sin ruido agregado (1):", cond1, "| B mejora Matthews con ruido 10 (2):", cond2)
print("→ se adopta:", ORDER[1] if cond1 and cond2 else ORDER[0])

B no inferior sin ruido agregado (1): True | B mejora Matthews con ruido 10 (2): True
→ se adopta: B. Wiener con ruido estimado


In [8]:
print("Matthews por grupo, sin ruido agregado y con ruido 10:")
R[R["ruido"].isin([0, 10])].join(dev["group"], on="image").pivot_table(
    index="candidata", columns=["ruido", "group"], values="matthews").loc[ORDER].round(3)

Matthews por grupo, sin ruido agregado y con ruido 10:


ruido                            0                    10              
group                            dr      g      h     dr      g      h
candidata                                                             
A. sin Wiener                 0.707  0.755  0.798  0.610  0.652  0.683
B. Wiener con ruido estimado  0.706  0.755  0.796  0.671  0.715  0.752
C. Wiener de la etapa 3       0.697  0.748  0.790  0.643  0.694  0.721

## Decisión (escrita después de correr)

**Control:** A sin ruido agregado reproduce exacto el Matthews de `05j` (0,7533). **El estimador del ruido acierta:** 2,50, 4,95 y 9,90 niveles de media con ruido de 5, 10 y 20 (lo esperado es 2,5, 5 y 10), y entre 0,49 y 0,74 en las fotos sin ruido agregado.

**Se adopta B, el Wiener con el ruido estimado** (`config.SEGMENT_BACKGROUND_MODE` con el paso "Wiener con ruido estimado"). Cumple las dos condiciones:

| B contra A, Matthews | Sin ruido agregado | Ruido 5 | Ruido 10 | Ruido 20 |
|---|---|---|---|---|
| A | 0,7533 | 0,7327 | 0,6482 | 0,2694 |
| B | 0,7525 | 0,7390 | 0,7123 | 0,6207 |
| Diferencia (intervalo 95 %) | −0,0008 (−0,0015 a −0,0002) | +0,006 (+0,003 a +0,009) | **+0,064 (+0,047 a +0,081)** | +0,351 (+0,296 a +0,398) |
| Imágenes que mejoran | 6 de 30 | 27 de 30 | 30 de 30 | 30 de 30 |

1. **Sin ruido agregado, B no es inferior**, pero **pierde un poco y se nota:** Matthews −0,0008 (−0,0015 a −0,0002), vasos finos −0,7 puntos (−1,3 a −0,3), Dice de línea central −0,0016; la precisión sube 0,2 puntos. Todo dentro del margen de −0,03. Es el precio de filtrar: aunque el ruido estimado es chico (0,5 niveles), el filtro suaviza un poco los vasos más finos y débiles.
2. **Con ruido 10, B mejora Matthews +0,064 en las 30 imágenes**, y también el Dice de línea central (+0,090), los vasos finos (+1,9 puntos) y la precisión (+3,1 puntos). Con ruido 20 pasa de 0,27 a 0,62.

**Por qué con el ruido estimado y no siempre (C).** El Wiener de la etapa 3, con el ruido de scipy, pierde 0,008 de Matthews y 4,3 puntos de vasos finos en las fotos sin ruido agregado (once veces más que B) y recupera menos con ruido (0,686 con ruido 10 y 0,563 con 20, contra 0,712 y 0,621 de B). Estimar el ruido de cada foto permite filtrar poco cuando la foto es buena y mucho cuando es mala.

**Hallazgo: las mejoras de esta ronda habían vuelto al algoritmo más sensible al ruido fuerte.** En la etapa 8, el algoritmo congelado (resta, 26 mapas) daba 0,73 sin ruido, cerca de 0,66 con ruido 10 y 0,41 con ruido 20. El de esta ronda antes del Wiener (A) da 0,75, 0,65 y 0,27. No se aisló la causa; la sospecha es la división por el fondo, que en las zonas oscuras también amplifica el ruido. Con B, el algoritmo queda mejor que el congelado en todas las condiciones (0,75, 0,71 y 0,62). Son corridas distintas (otros pesos y otro ruido sorteado), así que esta comparación es descriptiva.

**Cuidados.** En las fotos de la base (y en prueba, que son de la misma base) B cuesta cerca de una milésima de Matthews: se adopta por robustez, no para ganar en la base. Es la quinta ronda decidida con las mismas 30 imágenes; acá el efecto principal es grande y se mide con ruido agregado, así que el optimismo de elegir varias veces pesa poco. El conjunto de prueba no se usó.